In [1]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 17.9 MB/s  0:00:20
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 55.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [tensorflow]


In [2]:
# ==============================================================================
# CELL 1: SENSOR-FUSED TRAINING & SIMULATION PIPELINE (RUN ONCE)
# Synthesizes RCS-based uncertainty and biologically accurate flight kinematics.
# ==============================================================================
import numpy as np
import pandas as pd
import random
import joblib
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, Dense, Dropout, BatchNormalization,
                                     LayerNormalization, MultiHeadAttention, Masking,
                                     Bidirectional, GRU, concatenate, GlobalAveragePooling1D,
                                     GlobalMaxPooling1D, Add)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

CLASSES = [
    "CIVILIAN_AIRCRAFT", "FIGHTER_AIRCRAFT", "HELICOPTER", "DRONE",
    "SURFACE_MISSILE", "AERIAL_MISSILE", "SKIFF", "BOAT", "SHIP"
]

def robust_feature_extractor(df):
    """Core Physics & Uncertainty Fusion Engine"""
    df = df.copy()
    if 'track_no' not in df.columns: df['track_no'] = 1
    if 'ntp_sec' not in df.columns: df['ntp_sec'] = np.arange(len(df)) * 10.0

    df = df.sort_values(['track_no', 'ntp_sec']).reset_index(drop=True)
    df['dt'] = df.groupby('track_no')['ntp_sec'].diff().fillna(10.0).replace(0, 10.0)
    dt = df['dt'].values

    for axis in ['x', 'y', 'z']:
        if f'vel_{axis}' not in df.columns:
            df[f'vel_{axis}'] = df.groupby('track_no')[f'pos_{axis}'].diff() / df['dt']
        df[f'vel_{axis}'] = df[f'vel_{axis}'].fillna(0)
        if f'accel_{axis}' not in df.columns:
            df[f'accel_{axis}'] = df.groupby('track_no')[f'vel_{axis}'].diff() / df['dt']
        df[f'accel_{axis}'] = df[f'accel_{axis}'].fillna(0)

    df['vel_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    df['vel_xy_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2)
    if 'range_mag' not in df.columns: df['range_mag'] = np.sqrt(df['pos_x']**2 + df['pos_y']**2 + df['pos_z']**2)
    df['accel_mag'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)

    df['jerk_x'] = df.groupby('track_no')['accel_x'].diff() / dt
    df['jerk_y'] = df.groupby('track_no')['accel_y'].diff() / dt
    df['jerk_z'] = df.groupby('track_no')['accel_z'].diff() / dt
    df[['jerk_x', 'jerk_y', 'jerk_z']] = df[['jerk_x', 'jerk_y', 'jerk_z']].fillna(0)
    df['jerk_mag'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)
    df['jerk_to_accel_ratio'] = df['jerk_mag'] / (df['accel_mag'] + 1e-5)

    v, a, j = df[['vel_x', 'vel_y', 'vel_z']].values, df[['accel_x', 'accel_y', 'accel_z']].values, df[['jerk_x', 'jerk_y', 'jerk_z']].values
    v_cross_a = np.cross(v, a)
    v_cross_a_mag = np.linalg.norm(v_cross_a, axis=1)

    df['curvature_3d'] = np.nan_to_num(v_cross_a_mag / (df['vel_mag'].values**3 + 1e-6))
    df['torsion_3d'] = np.nan_to_num(np.einsum('ij,ij->i', v_cross_a, j) / (v_cross_a_mag**2 + 1e-6))

    df['specific_energy'] = 9.81 * df['pos_z'] + 0.5 * (df['vel_mag']**2)
    df['energy_rate'] = 9.81 * df['vel_z'] + (df['vel_x']*df['accel_x'] + df['vel_y']*df['accel_y'] + df['vel_z']*df['accel_z'])
    df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)

    for col in ['accel_mag', 'jerk_mag', 'energy_rate']:
        df[f'{col}_roll_mean'] = df.groupby('track_no')[col].transform(lambda x: x.rolling(3, min_periods=1).mean())
        df[f'{col}_roll_std'] = df.groupby('track_no')[col].transform(lambda x: x.rolling(3, min_periods=1).std()).fillna(0.0)

    df['cum_max_accel'] = df.groupby('track_no')['accel_mag'].cummax()
    df['cum_max_jerk'] = df.groupby('track_no')['jerk_mag'].cummax()
    df['cum_max_energy_rate'] = df.groupby('track_no')['energy_rate'].cummax()

    expected_uncertainty_cols = [
        'std_dev_x', 'std_dev_y', 'std_dev_z',
        'std_dev_vx', 'std_dev_vy', 'std_dev_vz',
        'cov_x_vx', 'cov_y_vy', 'cov_z_vz'
    ]

    log_feature_names = []
    for col in expected_uncertainty_cols:
        if col not in df.columns: df[col] = 1e6
        else: df[col] = df[col].fillna(1e6)
        df[f'{col}_log'] = np.log1p(np.abs(df[col])) * np.sign(df[col])
        log_feature_names.append(f'{col}_log')

    df = df.fillna(0.0)
    required_features = [
        'pos_z', 'vel_mag', 'vel_xy_mag', 'range_mag', 'accel_mag', 'jerk_mag',
        'jerk_to_accel_ratio', 'curvature_3d', 'torsion_3d', 'specific_energy',
        'energy_rate', 'z_ratio', 'vertical_accel_ratio',
        'accel_mag_roll_mean', 'accel_mag_roll_std',
        'jerk_mag_roll_mean', 'jerk_mag_roll_std',
        'energy_rate_roll_mean', 'energy_rate_roll_std',
        'cum_max_accel', 'cum_max_jerk', 'cum_max_energy_rate'
    ] + log_feature_names

    return df, required_features

def generate_complex_trajectory(track_class, duration=40, dt=10.0):
    """Generates tracks based on verified aerospace physics and human biology limitations."""
    pos_x_list, pos_y_list, pos_z_list = [], [], []
    vel_x_list, vel_y_list, vel_z_list = [], [], []
    accel_x_list, accel_y_list, accel_z_list = [], [], []
    track_no_list, track_class_list, ntp_sec_list = [], [], []

    std_x_list, std_y_list, std_z_list = [], [], []
    std_vx_list, std_vy_list, std_vz_list = [], [], []
    cov_x_vx_list, cov_y_vy_list, cov_z_vz_list = [], [], []

    track_id = random.randint(100000, 999999)
    angle = random.uniform(0, 2 * np.pi)
    pos_x, pos_y = 30000 * np.cos(angle), 30000 * np.sin(angle)

    # ACCURATE REAL-WORLD KINEMATIC ENVELOPES
    if track_class == "CIVILIAN_AIRCRAFT":
        # Cruising altitude 28k to 43k feet, speeds around Mach 0.8
        pos_z, v_mag = np.random.uniform(8500, 13100), np.random.uniform(220, 280)
        max_accel = 2.0 # Standard rate turns, max 0.2G for passenger comfort
    elif track_class == "FIGHTER_AIRCRAFT":
        # Deck level to service ceiling, transonic/supersonic
        pos_z, v_mag = np.random.uniform(100, 18000), np.random.uniform(250, 800)
        max_accel = 88.0 # Peak 9G maneuvers (Biological human limit)
    elif track_class == "HELICOPTER":
        pos_z, v_mag = np.random.uniform(10, 3000), np.random.uniform(0, 100)
        max_accel = 15.0 # Max ~1.5G
    elif track_class == "DRONE":
        pos_z, v_mag = np.random.uniform(10, 18000), np.random.uniform(0, 150)
        max_accel = 40.0 # High variability, unmanned frame structural limits
    elif track_class == "SURFACE_MISSILE":
        # Includes sea-skimmers (10-50m) and high ballistic trajectories
        pos_z = np.random.uniform(10, 50) if random.random() < 0.5 else np.random.uniform(10000, 25000)
        v_mag, max_accel = np.random.uniform(270, 1500), 150.0
    elif track_class == "AERIAL_MISSILE":
        pos_z, v_mag = np.random.uniform(2000, 25000), np.random.uniform(600, 1500)
        max_accel = 390.0 # Extreme terminal intercept phase (up to 40G)
    elif track_class == "SKIFF":
        pos_z, v_mag, max_accel = 0.0, np.random.uniform(10, 30), 5.0
    elif track_class == "BOAT":
        pos_z, v_mag, max_accel = 0.0, np.random.uniform(5, 15), 2.0
    elif track_class == "SHIP":
        pos_z, v_mag, max_accel = 0.0, np.random.uniform(1, 10), 0.5

    vel = np.array([v_mag * np.cos(angle), v_mag * np.sin(angle), 0.0])
    start_time = 1700000000

    # BASE RADAR CROSS SECTION (RCS) NOISE
    rcs_noise_base = {
        "CIVILIAN_AIRCRAFT": 1400, "SHIP": 1400,
        "HELICOPTER": 2500, "BOAT": 2500,
        "FIGHTER_AIRCRAFT": 4500, "AERIAL_MISSILE": 5500,
        "DRONE": 7000, "SKIFF": 7000, "SURFACE_MISSILE": 8000
    }

    for t in range(duration):
        if track_class in ["SKIFF", "BOAT", "SHIP"]: vel[2] = 0.0

        # Apply class-specific physical maneuvering limits
        accel_scale = max_accel if random.random() < 0.1 else (max_accel * 0.1)
        accel = np.random.normal(0, accel_scale / 3.0, 3)

        vel += accel * dt
        pos_x += vel[0] * dt; pos_y += vel[1] * dt; pos_z = max(0.0, pos_z + vel[2] * dt)

        base_noise = rcs_noise_base[track_class]
        sx = base_noise * random.uniform(0.7, 1.3)
        sy = base_noise * random.uniform(0.7, 1.3)
        sz = base_noise * random.uniform(0.7, 1.3)
        svx = sx / 15.0; svy = sy / 15.0; svz = sz / 15.0

        pos_x_list.append(pos_x); pos_y_list.append(pos_y); pos_z_list.append(pos_z)
        vel_x_list.append(vel[0]); vel_y_list.append(vel[1]); vel_z_list.append(vel[2])
        accel_x_list.append(accel[0]); accel_y_list.append(accel[1]); accel_z_list.append(accel[2])

        std_x_list.append(sx); std_y_list.append(sy); std_z_list.append(sz)
        std_vx_list.append(svx); std_vy_list.append(svy); std_vz_list.append(svz)
        cov_x_vx_list.append((sx**2) * random.uniform(0.1, 0.5) * random.choice([1, -1]))
        cov_y_vy_list.append((sy**2) * random.uniform(0.1, 0.5) * random.choice([1, -1]))
        cov_z_vz_list.append((sz**2) * random.uniform(0.1, 0.5) * random.choice([1, -1]))

        track_no_list.append(track_id); track_class_list.append(track_class); ntp_sec_list.append(start_time + int(t * dt))

    return pd.DataFrame({
        'pos_x': pos_x_list, 'pos_y': pos_y_list, 'pos_z': pos_z_list,
        'vel_x': vel_x_list, 'vel_y': vel_y_list, 'vel_z': vel_z_list,
        'accel_x': accel_x_list, 'accel_y': accel_y_list, 'accel_z': accel_z_list,
        'std_dev_x': std_x_list, 'std_dev_y': std_y_list, 'std_dev_z': std_z_list,
        'std_dev_vx': std_vx_list, 'std_dev_vy': std_vy_list, 'std_dev_vz': std_vz_list,
        'cov_x_vx': cov_x_vx_list, 'cov_y_vy': cov_y_vy_list, 'cov_z_vz': cov_z_vz_list,
        'track_no': track_no_list, 'track_class': track_class_list, 'ntp_sec': ntp_sec_list
    })

def build_robust_transformer(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    masked = Masking(mask_value=0.0)(inputs)

    c1 = Conv1D(32, kernel_size=3, padding='same', activation='relu')(masked)
    c2 = Conv1D(32, kernel_size=5, padding='same', activation='relu')(masked)
    x = BatchNormalization()(concatenate([c1, c2]))

    gru_out = Bidirectional(GRU(64, return_sequences=True, dropout=0.3))(x)
    attn_out = MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.2)(gru_out, gru_out)
    x = LayerNormalization()(gru_out + attn_out)

    pooled = concatenate([GlobalAveragePooling1D()(x), GlobalMaxPooling1D()(x)])
    dense = Dropout(0.5)(BatchNormalization()(Dense(128, activation='relu')(pooled)))
    outputs = Dense(num_classes, activation='softmax')(dense)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

if __name__ == "__main__":
    if not os.path.exists("trained_model"): os.makedirs("trained_model")

    print("[1/4] Generating 9-Class Trajectories with Aerospace Constraints...")
    dfs = [generate_complex_trajectory(cls) for cls in CLASSES for _ in range(150)]
    df_raw = pd.concat(dfs, ignore_index=True)

    print("[2/4] Fusing Kinematics with Kalman Covariance Noise...")
    df_enhanced, feature_cols = robust_feature_extractor(df_raw)

    scaler = StandardScaler()
    df_enhanced[feature_cols] = scaler.fit_transform(df_enhanced[feature_cols])
    joblib.dump(scaler, "trained_model/feature_scaler.pkl")
    joblib.dump(feature_cols, "trained_model/feature_columns_list.pkl")

    seq_length = 30
    track_sequences, track_labels = [], []
    for track_id, group in df_enhanced.groupby('track_no'):
        data_matrix = group.sort_values('ntp_sec')[feature_cols].values
        label = group['track_class'].iloc[0]
        for i in range(1, len(data_matrix) + 1):
            if i <= seq_length: track_sequences.append(np.pad(data_matrix[:i], ((seq_length - i, 0), (0, 0)), 'constant'))
            else: track_sequences.append(data_matrix[i-seq_length:i])
            track_labels.append(label)

    X, y = np.array(track_sequences), np.array(track_labels)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    joblib.dump(le, "trained_model/label_encoder.pkl")

    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

    print("[3/4] Building Multi-Head Attention Model...")
    model = build_robust_transformer((X.shape[1], X.shape[2]), len(le.classes_))

    print("[4/4] Training Model (Early stopping enabled)...")
    model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=64,
              callbacks=[EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)])

    model.save("trained_model/radar_target_classifier.keras")
    print("\n✅ Training Complete. SOTA model and artifacts saved to 'trained_model/'.")

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


[1/4] Generating 9-Class Trajectories with Aerospace Constraints...
[2/4] Fusing Kinematics with Kalman Covariance Noise...
[3/4] Building Multi-Head Attention Model...
[4/4] Training Model (Early stopping enabled)...
Epoch 1/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 30s 40ms/step - accuracy: 0.8737 - loss: 0.3447 - val_accuracy: 0.9784 - val_loss: 0.0624
Epoch 2/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 26s 38ms/step - accuracy: 0.9701 - loss: 0.0863 - val_accuracy: 0.9895 - val_loss: 0.0260
Epoch 3/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 26s 38ms/step - accuracy: 0.9807 - loss: 0.0560 - val_accuracy: 0.9907 - val_loss: 0.0261
Epoch 4/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 26s 38ms/step - accuracy: 0.9858 - loss: 0.0405 - val_accuracy: 0.9913 - val_loss: 0.0227
Epoch 5/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 26s 38ms/step - accuracy: 0.9873 - loss: 0.0373 - val_accuracy: 0.9954 - val_loss: 0.0135
Epoch 6/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 25s 38ms/step - accuracy: 0.9896 - loss: 0.0317 - val_accuracy: 0.9930 - val_loss: 0.0192
Ep

In [3]:
# ==============================================================================
# CELL 2: LIVE INFERENCE BRIDGE (STANDALONE DEPLOYMENT SCRIPT)
# Processes raw unlabelled telemetry strictly using the trained AI artifacts.
# ==============================================================================
import pandas as pd
import numpy as np
import os
import joblib
from tensorflow.keras.models import load_model
import warnings
warnings.filterwarnings('ignore')

def robust_feature_extractor(df):
    """Exact duplicate of training extractor to ensure standalone production capability."""
    df = df.copy()
    if 'track_no' not in df.columns: df['track_no'] = 1
    if 'ntp_sec' not in df.columns: df['ntp_sec'] = np.arange(len(df)) * 10.0

    df = df.sort_values(['track_no', 'ntp_sec']).reset_index(drop=True)
    df['dt'] = df.groupby('track_no')['ntp_sec'].diff().fillna(10.0).replace(0, 10.0)
    dt = df['dt'].values

    for axis in ['x', 'y', 'z']:
        if f'vel_{axis}' not in df.columns: df[f'vel_{axis}'] = df.groupby('track_no')[f'pos_{axis}'].diff() / df['dt']
        df[f'vel_{axis}'] = df[f'vel_{axis}'].fillna(0)
        if f'accel_{axis}' not in df.columns: df[f'accel_{axis}'] = df.groupby('track_no')[f'vel_{axis}'].diff() / df['dt']
        df[f'accel_{axis}'] = df[f'accel_{axis}'].fillna(0)

    df['vel_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    df['vel_xy_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2)
    if 'range_mag' not in df.columns: df['range_mag'] = np.sqrt(df['pos_x']**2 + df['pos_y']**2 + df['pos_z']**2)
    df['accel_mag'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)

    df['jerk_x'] = df.groupby('track_no')['accel_x'].diff() / dt
    df['jerk_y'] = df.groupby('track_no')['accel_y'].diff() / dt
    df['jerk_z'] = df.groupby('track_no')['accel_z'].diff() / dt
    df[['jerk_x', 'jerk_y', 'jerk_z']] = df[['jerk_x', 'jerk_y', 'jerk_z']].fillna(0)
    df['jerk_mag'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)
    df['jerk_to_accel_ratio'] = df['jerk_mag'] / (df['accel_mag'] + 1e-5)

    v, a, j = df[['vel_x', 'vel_y', 'vel_z']].values, df[['accel_x', 'accel_y', 'accel_z']].values, df[['jerk_x', 'jerk_y', 'jerk_z']].values
    v_cross_a = np.cross(v, a)
    v_cross_a_mag = np.linalg.norm(v_cross_a, axis=1)
    df['curvature_3d'] = np.nan_to_num(v_cross_a_mag / (df['vel_mag'].values**3 + 1e-6))
    df['torsion_3d'] = np.nan_to_num(np.einsum('ij,ij->i', v_cross_a, j) / (v_cross_a_mag**2 + 1e-6))

    df['specific_energy'] = 9.81 * df['pos_z'] + 0.5 * (df['vel_mag']**2)
    df['energy_rate'] = 9.81 * df['vel_z'] + (df['vel_x']*df['accel_x'] + df['vel_y']*df['accel_y'] + df['vel_z']*df['accel_z'])
    df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)

    for col in ['accel_mag', 'jerk_mag', 'energy_rate']:
        df[f'{col}_roll_mean'] = df.groupby('track_no')[col].transform(lambda x: x.rolling(3, min_periods=1).mean())
        df[f'{col}_roll_std'] = df.groupby('track_no')[col].transform(lambda x: x.rolling(3, min_periods=1).std()).fillna(0.0)

    df['cum_max_accel'] = df.groupby('track_no')['accel_mag'].cummax()
    df['cum_max_jerk'] = df.groupby('track_no')['jerk_mag'].cummax()
    df['cum_max_energy_rate'] = df.groupby('track_no')['energy_rate'].cummax()

    expected_uncertainty_cols = [
        'std_dev_x', 'std_dev_y', 'std_dev_z',
        'std_dev_vx', 'std_dev_vy', 'std_dev_vz',
        'cov_x_vx', 'cov_y_vy', 'cov_z_vz'
    ]

    log_feature_names = []
    for col in expected_uncertainty_cols:
        if col not in df.columns: df[col] = 1e6
        else: df[col] = df[col].fillna(1e6)
        df[f'{col}_log'] = np.log1p(np.abs(df[col])) * np.sign(df[col])
        log_feature_names.append(f'{col}_log')

    df = df.fillna(0.0)
    required_features = [
        'pos_z', 'vel_mag', 'vel_xy_mag', 'range_mag', 'accel_mag', 'jerk_mag',
        'jerk_to_accel_ratio', 'curvature_3d', 'torsion_3d', 'specific_energy',
        'energy_rate', 'z_ratio', 'vertical_accel_ratio',
        'accel_mag_roll_mean', 'accel_mag_roll_std',
        'jerk_mag_roll_mean', 'jerk_mag_roll_std',
        'energy_rate_roll_mean', 'energy_rate_roll_std',
        'cum_max_accel', 'cum_max_jerk', 'cum_max_energy_rate'
    ] + log_feature_names

    return df, required_features


class LiveTrackManager:
    def __init__(self):
        print("Loading Deployed AI Configuration...")
        self.model = load_model("trained_model/radar_target_classifier.keras", compile=False)
        self.le = joblib.load("trained_model/label_encoder.pkl")
        self.scaler = joblib.load("trained_model/feature_scaler.pkl")
        self.feature_cols = joblib.load("trained_model/feature_columns_list.pkl")
        self.active_tracks = {}
        self.seq_length = 30

    def process_packet(self, raw_packet):
        tid = raw_packet['track_no']
        if tid not in self.active_tracks: self.active_tracks[tid] = []

        self.active_tracks[tid].append(raw_packet)
        if len(self.active_tracks[tid]) > 100: self.active_tracks[tid] = self.active_tracks[tid][-100:]

        try:
            track_df = pd.DataFrame(self.active_tracks[tid])
            features_df, _ = robust_feature_extractor(track_df)

            for col in self.feature_cols:
                if col not in features_df.columns: features_df[col] = 0.0

            features_scaled = self.scaler.transform(features_df[self.feature_cols].values)

            if len(features_scaled) >= self.seq_length:
                input_data = features_scaled[-self.seq_length:].reshape(1, self.seq_length, -1)
            else:
                input_data = np.pad(features_scaled, ((self.seq_length - len(features_scaled), 0), (0, 0)), 'constant').reshape(1, self.seq_length, -1)

            pred_probs = self.model.predict(input_data, verbose=0)[0]
            confidence = float(np.max(pred_probs))
            predicted_label = self.le.inverse_transform([np.argmax(pred_probs)])[0]

            return {'track_id': tid, 'prediction': predicted_label, 'confidence': confidence, 'packets': len(self.active_tracks[tid])}

        except Exception as e:
            return {'track_id': tid, 'prediction': f'ERROR: {str(e)}', 'confidence': 0.0, 'packets': 0}

if __name__ == "__main__":
    TEST_FILE = "/content/radar_data.csv"

    if os.path.exists(TEST_FILE):
        manager = LiveTrackManager()
        raw_stream = pd.read_csv(TEST_FILE).dropna(subset=['track_no']).sort_values('ntp_sec')
        target_ids_to_watch = list(raw_stream['track_no'].unique()[:3])

        print("\n" + "="*75)
        print(f" DEPLOYED RADAR INFERENCE ENGINE - STREAMING FROM: {TEST_FILE}")
        print("="*75)

        for packet in raw_stream.to_dict('records'):
            result = manager.process_packet(packet)

            if result['track_id'] in target_ids_to_watch:
                # Raw ICD tracking label (Kept completely uninterpreted by the script)
                raw_icd_tag = packet.get('track_class', 'UNLABELLED')
                if pd.isna(raw_icd_tag):
                    raw_icd_tag = "UNLABELLED"
                else:
                    raw_icd_tag = str(int(raw_icd_tag)) if isinstance(raw_icd_tag, float) else str(raw_icd_tag)

                print(f"📡 [Time: {packet['ntp_sec']}] Track {result['track_id']} (Hardware ICD Tag: {raw_icd_tag})")
                print(f"   -> AI Classification: {result['prediction']:<18s} | AI Confidence: {result['confidence']:.2%}\n")
    else:
        print(f"❌ Could not find {TEST_FILE}.")

Loading Deployed AI Configuration...

 DEPLOYED RADAR INFERENCE ENGINE - STREAMING FROM: radar_data.csv
📡 [Time: 3991182426] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: HELICOPTER         | AI Confidence: 94.83%

📡 [Time: 3991182434] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: SURFACE_MISSILE    | AI Confidence: 98.83%

📡 [Time: 3991182444] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: DRONE              | AI Confidence: 77.94%

📡 [Time: 3991182454] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: DRONE              | AI Confidence: 96.01%

📡 [Time: 3991182464] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: DRONE              | AI Confidence: 97.21%

📡 [Time: 3991182474] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: DRONE              | AI Confidence: 95.71%

📡 [Time: 3991182484] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: DRONE              | AI Confidence: 95.88%

📡 [Time: 3991182494] Track 1 (Hardware ICD Tag: 5)
   ->

## **Stress test**

In [4]:
# ==============================================================================
# CELL 3: THE ULTIMATE STRESS TEST (DISGUISES + BLACKOUTS + NOISE)
# Run this to evaluate the robustness of your trained model.
# ==============================================================================
import pandas as pd
import numpy as np
import random
import os
import joblib
from tensorflow.keras.models import load_model
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. EXTREME STRESS TRAJECTORY GENERATOR
# ---------------------------------------------------------
def generate_extreme_stress_trajectory(true_class, duration=40, dt=10.0):
    pos_x, pos_y = np.random.uniform(-30000, 30000, 2)
    angle = random.uniform(0, 2 * np.pi)
    track_id = random.randint(1000, 9999)
    start_time = 1700000000

    data = []

    # Force a disguise scenario for the stress test
    is_disguised = true_class in ["FIGHTER_AIRCRAFT", "SURFACE_MISSILE", "DRONE"]
    active_class = "CIVILIAN_AIRCRAFT" if true_class == "FIGHTER_AIRCRAFT" else ("BOAT" if true_class == "SURFACE_MISSILE" else ("HELICOPTER" if true_class == "DRONE" else true_class))

    if active_class == "CIVILIAN_AIRCRAFT": pos_z, v_mag = 9500, 250
    elif active_class == "HELICOPTER": pos_z, v_mag = 1000, 50
    elif active_class == "BOAT": pos_z, v_mag = 0.0, 10
    else: pos_z, v_mag = 5000, 300

    vel = np.array([v_mag * np.cos(angle), v_mag * np.sin(angle), 0.0])

    rcs_noise_base = {
        "CIVILIAN_AIRCRAFT": 1400, "SHIP": 1400, "HELICOPTER": 2500, "BOAT": 2500,
        "FIGHTER_AIRCRAFT": 4500, "AERIAL_MISSILE": 5500, "DRONE": 7000, "SKIFF": 7000, "SURFACE_MISSILE": 8000
    }

    in_blackout = False

    for t in range(duration):
        # 1. DISGUISE BREAKOUT (Happens at t=20)
        if is_disguised and t == 20:
            is_disguised = False
            active_class = true_class
            if true_class == "FIGHTER_AIRCRAFT":
                vel *= 2.0  # Instantly go supersonic
                vel[2] -= 250 # Extreme evasive dive
            elif true_class == "SURFACE_MISSILE":
                vel[2] += 300 # Sea-skimmer terminal pop-up
            elif true_class == "DRONE":
                vel = np.array([-vel[1], vel[0], vel[2]]) # Impossible 90-degree pivot

        # 2. BURST PACKET LOSS (Radar Blackout around the maneuver)
        if t in [21, 22, 23]: # Radar loses lock right after the crazy maneuver!
            continue

        # 3. PHYSICS & KINEMATICS
        if active_class in ["SKIFF", "BOAT", "SHIP"]: vel[2] = 0.0
        accel_scale = 88.0 if active_class == "FIGHTER_AIRCRAFT" else (150.0 if "MISSILE" in active_class else 2.0)
        accel = np.random.normal(0, accel_scale / (10.0 if not is_disguised else 30.0), 3)
        vel += accel * dt

        pos_x += vel[0] * dt; pos_y += vel[1] * dt; pos_z = max(0.0, pos_z + vel[2] * dt)

        # 4. COVARIANCE EXPLOSION
        # If we just recovered from a blackout, uncertainty spikes massively
        base_noise = rcs_noise_base[true_class]
        noise_multiplier = 5.0 if t == 24 else 1.0

        sx = base_noise * random.uniform(0.7, 1.3) * noise_multiplier
        sy = base_noise * random.uniform(0.7, 1.3) * noise_multiplier
        sz = base_noise * random.uniform(0.7, 1.3) * noise_multiplier
        svx, svy, svz = sx/15.0, sy/15.0, sz/15.0

        data.append({
            'ntp_sec': start_time + int(t * dt),
            'track_no': track_id, 'track_class': true_class, 'disguised_as': active_class,
            'pos_x': pos_x, 'pos_y': pos_y, 'pos_z': pos_z,
            'vel_x': vel[0], 'vel_y': vel[1], 'vel_z': vel[2],
            'accel_x': accel[0], 'accel_y': accel[1], 'accel_z': accel[2],
            'std_dev_x': sx, 'std_dev_y': sy, 'std_dev_z': sz,
            'std_dev_vx': svx, 'std_dev_vy': svy, 'std_dev_vz': svz,
            'cov_x_vx': (sx**2) * random.choice([1, -1]),
            'cov_y_vy': (sy**2) * random.choice([1, -1]),
            'cov_z_vz': (sz**2) * random.choice([1, -1])
        })
    return pd.DataFrame(data)


# ---------------------------------------------------------
# 2. FEATURE EXTRACTOR (Must match training exactly)
# ---------------------------------------------------------
def robust_feature_extractor(df):
    df = df.copy()
    if 'track_no' not in df.columns: df['track_no'] = 1
    if 'ntp_sec' not in df.columns: df['ntp_sec'] = np.arange(len(df)) * 10.0

    df = df.sort_values(['track_no', 'ntp_sec']).reset_index(drop=True)
    df['dt'] = df.groupby('track_no')['ntp_sec'].diff().fillna(10.0).replace(0, 10.0)
    dt = df['dt'].values

    for axis in ['x', 'y', 'z']:
        if f'vel_{axis}' not in df.columns: df[f'vel_{axis}'] = df.groupby('track_no')[f'pos_{axis}'].diff() / df['dt']
        df[f'vel_{axis}'] = df[f'vel_{axis}'].fillna(0)
        if f'accel_{axis}' not in df.columns: df[f'accel_{axis}'] = df.groupby('track_no')[f'vel_{axis}'].diff() / df['dt']
        df[f'accel_{axis}'] = df[f'accel_{axis}'].fillna(0)

    df['vel_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    df['vel_xy_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2)
    if 'range_mag' not in df.columns: df['range_mag'] = np.sqrt(df['pos_x']**2 + df['pos_y']**2 + df['pos_z']**2)
    df['accel_mag'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)

    df['jerk_x'] = df.groupby('track_no')['accel_x'].diff() / dt
    df['jerk_y'] = df.groupby('track_no')['accel_y'].diff() / dt
    df['jerk_z'] = df.groupby('track_no')['accel_z'].diff() / dt
    df[['jerk_x', 'jerk_y', 'jerk_z']] = df[['jerk_x', 'jerk_y', 'jerk_z']].fillna(0)
    df['jerk_mag'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)
    df['jerk_to_accel_ratio'] = df['jerk_mag'] / (df['accel_mag'] + 1e-5)

    v, a, j = df[['vel_x', 'vel_y', 'vel_z']].values, df[['accel_x', 'accel_y', 'accel_z']].values, df[['jerk_x', 'jerk_y', 'jerk_z']].values
    v_cross_a = np.cross(v, a)
    v_cross_a_mag = np.linalg.norm(v_cross_a, axis=1)
    df['curvature_3d'] = np.nan_to_num(v_cross_a_mag / (df['vel_mag'].values**3 + 1e-6))
    df['torsion_3d'] = np.nan_to_num(np.einsum('ij,ij->i', v_cross_a, j) / (v_cross_a_mag**2 + 1e-6))

    df['specific_energy'] = 9.81 * df['pos_z'] + 0.5 * (df['vel_mag']**2)
    df['energy_rate'] = 9.81 * df['vel_z'] + (df['vel_x']*df['accel_x'] + df['vel_y']*df['accel_y'] + df['vel_z']*df['accel_z'])
    df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)

    for col in ['accel_mag', 'jerk_mag', 'energy_rate']:
        df[f'{col}_roll_mean'] = df.groupby('track_no')[col].transform(lambda x: x.rolling(3, min_periods=1).mean())
        df[f'{col}_roll_std'] = df.groupby('track_no')[col].transform(lambda x: x.rolling(3, min_periods=1).std()).fillna(0.0)

    df['cum_max_accel'] = df.groupby('track_no')['accel_mag'].cummax()
    df['cum_max_jerk'] = df.groupby('track_no')['jerk_mag'].cummax()
    df['cum_max_energy_rate'] = df.groupby('track_no')['energy_rate'].cummax()

    expected_uncertainty_cols = ['std_dev_x', 'std_dev_y', 'std_dev_z', 'std_dev_vx', 'std_dev_vy', 'std_dev_vz', 'cov_x_vx', 'cov_y_vy', 'cov_z_vz']
    log_feature_names = []
    for col in expected_uncertainty_cols:
        if col not in df.columns: df[col] = 1e6
        else: df[col] = df[col].fillna(1e6)
        df[f'{col}_log'] = np.log1p(np.abs(df[col])) * np.sign(df[col])
        log_feature_names.append(f'{col}_log')

    df = df.fillna(0.0)
    required_features = [
        'pos_z', 'vel_mag', 'vel_xy_mag', 'range_mag', 'accel_mag', 'jerk_mag',
        'jerk_to_accel_ratio', 'curvature_3d', 'torsion_3d', 'specific_energy',
        'energy_rate', 'z_ratio', 'vertical_accel_ratio', 'accel_mag_roll_mean', 'accel_mag_roll_std',
        'jerk_mag_roll_mean', 'jerk_mag_roll_std', 'energy_rate_roll_mean', 'energy_rate_roll_std',
        'cum_max_accel', 'cum_max_jerk', 'cum_max_energy_rate'
    ] + log_feature_names

    return df, required_features

# ---------------------------------------------------------
# 3. LIVE STREAM EXECUTION
# ---------------------------------------------------------
if __name__ == "__main__":
    print("Generating Extreme Stress Test Scenarios...")
    # Generate 3 highly deceptive tracks
    df1 = generate_extreme_stress_trajectory("FIGHTER_AIRCRAFT") # Pretends to be civilian
    df2 = generate_extreme_stress_trajectory("SURFACE_MISSILE")  # Pretends to be a boat

    stress_df = pd.concat([df1, df2]).sort_values('ntp_sec')

    print("Loading AI Artifacts...")
    model = load_model("trained_model/radar_target_classifier.keras", compile=False)
    le = joblib.load("trained_model/label_encoder.pkl")
    scaler = joblib.load("trained_model/feature_scaler.pkl")
    feature_cols = joblib.load("trained_model/feature_columns_list.pkl")

    active_tracks = {}
    seq_length = 30

    print("\n" + "="*85)
    print(" EXTREME STRESS TEST: DISGUISES, RADAR BLACKOUTS, COVARIANCE SPIKES")
    print("="*85)
    print("Watch the output closely. At packet ~20, the target drops its disguise.")
    print("Then the radar goes blind (packets skipped). Let's see if the AI recovers!\n")

    packet_counter = {tid: 0 for tid in stress_df['track_no'].unique()}

    for packet in stress_df.to_dict('records'):
        tid = packet['track_no']
        if tid not in active_tracks: active_tracks[tid] = []
        active_tracks[tid].append(packet)
        packet_counter[tid] += 1

        # Determine simulation stage for printing
        sim_stage = "CRUISING (DISGUISED)"
        if packet_counter[tid] == 21: sim_stage = "💥 BREAKOUT & BLACKOUT RECOVERY 💥"
        elif packet_counter[tid] > 21: sim_stage = "EVASIVE MANEUVERING"

        try:
            track_df = pd.DataFrame(active_tracks[tid])
            features_df, _ = robust_feature_extractor(track_df)

            for col in feature_cols:
                if col not in features_df.columns: features_df[col] = 0.0

            features_scaled = scaler.transform(features_df[feature_cols].values)

            if len(features_scaled) >= seq_length:
                input_data = features_scaled[-seq_length:].reshape(1, seq_length, -1)
            else:
                input_data = np.pad(features_scaled, ((seq_length - len(features_scaled), 0), (0, 0)), 'constant').reshape(1, seq_length, -1)

            pred_probs = model.predict(input_data, verbose=0)[0]
            confidence = float(np.max(pred_probs))
            predicted_label = le.inverse_transform([np.argmax(pred_probs)])[0]

            # Print Output
            true_cls = packet['track_class']
            disguise_cls = packet['disguised_as']

            status = "🚨 EXPOSED" if predicted_label == true_cls else "🕵️ FOOLED"

            print(f"📡 [Pkt: {packet_counter[tid]:02d}] Track {tid} | State: {sim_stage}")
            print(f"   -> Ground Truth: {true_cls} (Acting like a {disguise_cls})")
            print(f"   -> AI Verdict:   {predicted_label:<18s} | Conf: {confidence:.2%} [{status}]\n")

        except Exception as e:
            print(f"Error: {e}")

Generating Extreme Stress Test Scenarios...
Loading AI Artifacts...

 EXTREME STRESS TEST: DISGUISES, RADAR BLACKOUTS, COVARIANCE SPIKES
Watch the output closely. At packet ~20, the target drops its disguise.
Then the radar goes blind (packets skipped). Let's see if the AI recovers!

📡 [Pkt: 01] Track 4775 | State: CRUISING (DISGUISED)
   -> Ground Truth: FIGHTER_AIRCRAFT (Acting like a CIVILIAN_AIRCRAFT)
   -> AI Verdict:   FIGHTER_AIRCRAFT   | Conf: 95.56% [🚨 EXPOSED]

📡 [Pkt: 01] Track 2913 | State: CRUISING (DISGUISED)
   -> Ground Truth: SURFACE_MISSILE (Acting like a BOAT)
   -> AI Verdict:   SKIFF              | Conf: 99.49% [🕵️ FOOLED]

📡 [Pkt: 02] Track 2913 | State: CRUISING (DISGUISED)
   -> Ground Truth: SURFACE_MISSILE (Acting like a BOAT)
   -> AI Verdict:   SKIFF              | Conf: 99.29% [🕵️ FOOLED]

📡 [Pkt: 02] Track 4775 | State: CRUISING (DISGUISED)
   -> Ground Truth: FIGHTER_AIRCRAFT (Acting like a CIVILIAN_AIRCRAFT)
   -> AI Verdict:   FIGHTER_AIRCRAFT   | Conf:

In [5]:
# ==============================================================================
# CELL 2: LIVE INFERENCE BRIDGE WITH EXECUTIVE MISSION SUMMARY
# Processes raw unlabelled telemetry, streams live updates, and outputs a
# comprehensive final summary table for all unique tracks found in the file.
# ==============================================================================
import pandas as pd
import numpy as np
import os
import joblib
from tensorflow.keras.models import load_model
import warnings
warnings.filterwarnings('ignore')

def robust_feature_extractor(df):
    """Exact duplicate of training extractor to ensure standalone production capability."""
    df = df.copy()
    if 'track_no' not in df.columns: df['track_no'] = 1
    if 'ntp_sec' not in df.columns: df['ntp_sec'] = np.arange(len(df)) * 10.0

    df = df.sort_values(['track_no', 'ntp_sec']).reset_index(drop=True)
    df['dt'] = df.groupby('track_no')['ntp_sec'].diff().fillna(10.0).replace(0, 10.0)
    dt = df['dt'].values

    for axis in ['x', 'y', 'z']:
        if f'vel_{axis}' not in df.columns: df[f'vel_{axis}'] = df.groupby('track_no')[f'pos_{axis}'].diff() / df['dt']
        df[f'vel_{axis}'] = df[f'vel_{axis}'].fillna(0)
        if f'accel_{axis}' not in df.columns: df[f'accel_{axis}'] = df.groupby('track_no')[f'vel_{axis}'].diff() / df['dt']
        df[f'accel_{axis}'] = df[f'accel_{axis}'].fillna(0)

    df['vel_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    df['vel_xy_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2)
    if 'range_mag' not in df.columns: df['range_mag'] = np.sqrt(df['pos_x']**2 + df['pos_y']**2 + df['pos_z']**2)
    df['accel_mag'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)

    df['jerk_x'] = df.groupby('track_no')['accel_x'].diff() / dt
    df['jerk_y'] = df.groupby('track_no')['accel_y'].diff() / dt
    df['jerk_z'] = df.groupby('track_no')['accel_z'].diff() / dt
    df[['jerk_x', 'jerk_y', 'jerk_z']] = df[['jerk_x', 'jerk_y', 'jerk_z']].fillna(0)
    df['jerk_mag'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)
    df['jerk_to_accel_ratio'] = df['jerk_mag'] / (df['accel_mag'] + 1e-5)

    v, a, j = df[['vel_x', 'vel_y', 'vel_z']].values, df[['accel_x', 'accel_y', 'accel_z']].values, df[['jerk_x', 'jerk_y', 'jerk_z']].values
    v_cross_a = np.cross(v, a)
    v_cross_a_mag = np.linalg.norm(v_cross_a, axis=1)
    df['curvature_3d'] = np.nan_to_num(v_cross_a_mag / (df['vel_mag'].values**3 + 1e-6))
    df['torsion_3d'] = np.nan_to_num(np.einsum('ij,ij->i', v_cross_a, j) / (v_cross_a_mag**2 + 1e-6))

    df['specific_energy'] = 9.81 * df['pos_z'] + 0.5 * (df['vel_mag']**2)
    df['energy_rate'] = 9.81 * df['vel_z'] + (df['vel_x']*df['accel_x'] + df['vel_y']*df['accel_y'] + df['vel_z']*df['accel_z'])
    df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)

    for col in ['accel_mag', 'jerk_mag', 'energy_rate']:
        df[f'{col}_roll_mean'] = df.groupby('track_no')[col].transform(lambda x: x.rolling(3, min_periods=1).mean())
        df[f'{col}_roll_std'] = df.groupby('track_no')[col].transform(lambda x: x.rolling(3, min_periods=1).std()).fillna(0.0)

    df['cum_max_accel'] = df.groupby('track_no')['accel_mag'].cummax()
    df['cum_max_jerk'] = df.groupby('track_no')['jerk_mag'].cummax()
    df['cum_max_energy_rate'] = df.groupby('track_no')['energy_rate'].cummax()

    expected_uncertainty_cols = [
        'std_dev_x', 'std_dev_y', 'std_dev_z',
        'std_dev_vx', 'std_dev_vy', 'std_dev_vz',
        'cov_x_vx', 'cov_y_vy', 'cov_z_vz'
    ]

    log_feature_names = []
    for col in expected_uncertainty_cols:
        if col not in df.columns: df[col] = 1e6
        else: df[col] = df[col].fillna(1e6)
        df[f'{col}_log'] = np.log1p(np.abs(df[col])) * np.sign(df[col])
        log_feature_names.append(f'{col}_log')

    df = df.fillna(0.0)
    required_features = [
        'pos_z', 'vel_mag', 'vel_xy_mag', 'range_mag', 'accel_mag', 'jerk_mag',
        'jerk_to_accel_ratio', 'curvature_3d', 'torsion_3d', 'specific_energy',
        'energy_rate', 'z_ratio', 'vertical_accel_ratio',
        'accel_mag_roll_mean', 'accel_mag_roll_std',
        'jerk_mag_roll_mean', 'jerk_mag_roll_std',
        'energy_rate_roll_mean', 'energy_rate_roll_std',
        'cum_max_accel', 'cum_max_jerk', 'cum_max_energy_rate'
    ] + log_feature_names

    return df, required_features


class LiveTrackManager:
    def __init__(self):
        print("Loading Deployed AI Configuration...")
        self.model = load_model("trained_model/radar_target_classifier.keras", compile=False)
        self.le = joblib.load("trained_model/label_encoder.pkl")
        self.scaler = joblib.load("trained_model/feature_scaler.pkl")
        self.feature_cols = joblib.load("trained_model/feature_columns_list.pkl")
        self.active_tracks = {}
        self.seq_length = 30

    def process_packet(self, raw_packet):
        tid = raw_packet['track_no']
        if tid not in self.active_tracks: self.active_tracks[tid] = []

        self.active_tracks[tid].append(raw_packet)
        if len(self.active_tracks[tid]) > 100: self.active_tracks[tid] = self.active_tracks[tid][-100:]

        try:
            track_df = pd.DataFrame(self.active_tracks[tid])
            features_df, _ = robust_feature_extractor(track_df)

            for col in self.feature_cols:
                if col not in features_df.columns: features_df[col] = 0.0

            features_scaled = self.scaler.transform(features_df[self.feature_cols].values)

            if len(features_scaled) >= self.seq_length:
                input_data = features_scaled[-self.seq_length:].reshape(1, self.seq_length, -1)
            else:
                input_data = np.pad(features_scaled, ((self.seq_length - len(features_scaled), 0), (0, 0)), 'constant').reshape(1, self.seq_length, -1)

            pred_probs = self.model.predict(input_data, verbose=0)[0]
            confidence = float(np.max(pred_probs))
            predicted_label = self.le.inverse_transform([np.argmax(pred_probs)])[0]

            return {'track_id': tid, 'prediction': predicted_label, 'confidence': confidence, 'packets': len(self.active_tracks[tid])}

        except Exception as e:
            return {'track_id': tid, 'prediction': f'ERROR: {str(e)}', 'confidence': 0.0, 'packets': 0}

if __name__ == "__main__":
    TEST_FILE = "/content/radar_data.csv"

    if os.path.exists(TEST_FILE):
        manager = LiveTrackManager()
        raw_stream = pd.read_csv(TEST_FILE).dropna(subset=['track_no']).sort_values('ntp_sec')

        # Get all unique tracks present in the file
        all_track_ids = raw_stream['track_no'].unique()
        print(f"\nFound {len(all_track_ids)} unique track(s) in {TEST_FILE}.")

        print("\n" + "="*75)
        print(f" DEPLOYED RADAR INFERENCE ENGINE - STREAMING FROM: {TEST_FILE}")
        print("="*75)

        # Track the final state for every unique track ID in the file
        track_final_results = {}

        for packet in raw_stream.to_dict('records'):
            result = manager.process_packet(packet)
            tid = result['track_id']

            # Continuously update the final state for this track as new packets arrive
            track_final_results[tid] = {
                'Track ID': tid,
                'Total Packets': result['packets'],
                'Final Classification': result['prediction'],
                'Final Confidence': f"{result['confidence']:.2%}"
            }

            # Optional: Print live updates for the first few tracks to keep console clean
            if tid in all_track_ids[:3]:
                raw_icd_tag = packet.get('track_class', 'UNLABELLED')
                if pd.isna(raw_icd_tag):
                    raw_icd_tag = "UNLABELLED"
                else:
                    raw_icd_tag = str(int(raw_icd_tag)) if isinstance(raw_icd_tag, float) else str(raw_icd_tag)

                print(f"📡 [Time: {packet['ntp_sec']}] Track {tid} (Hardware ICD Tag: {raw_icd_tag})")
                print(f"   -> AI Classification: {result['prediction']:<18s} | AI Confidence: {result['confidence']:.2%}\n")

        # ======================================================================
        # EXECUTIVE MISSION SUMMARY TABLE
        # ======================================================================
        summary_df = pd.DataFrame(list(track_final_results.values()))

        print("\n" + "="*75)
        print(" 🎯 POST-MISSION EXECUTIVE SUMMARY TABLE (ALL TRACKS)")
        print("="*75)
        print(summary_df.to_string(index=False))
        print("="*75)

    else:
        print(f"❌ Could not find {TEST_FILE}.")

Loading Deployed AI Configuration...

Found 1 unique track(s) in /content/radar_data.csv.

 DEPLOYED RADAR INFERENCE ENGINE - STREAMING FROM: /content/radar_data.csv
📡 [Time: 3991182426] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: HELICOPTER         | AI Confidence: 94.83%

📡 [Time: 3991182434] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: SURFACE_MISSILE    | AI Confidence: 98.83%

📡 [Time: 3991182444] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: DRONE              | AI Confidence: 77.94%

📡 [Time: 3991182454] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: DRONE              | AI Confidence: 96.01%

📡 [Time: 3991182464] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: DRONE              | AI Confidence: 97.21%

📡 [Time: 3991182474] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: DRONE              | AI Confidence: 95.71%

📡 [Time: 3991182484] Track 1 (Hardware ICD Tag: 5)
   -> AI Classification: DRONE              | AI Confidence: 95